# 04 Data Preprocessing

This notebook contains **Step 4** of the workflow:

- change-feature creation
- missing-value imputation
- categorical encoding
- train/test split preparation

This notebook expects the `model_features` table created in **03_Feature_Engineering.ipynb**.


In [ ]:
import duckdb
import os
import pandas as pd

from utils import get_db_connection
con = get_db_connection()

# Confirm tables are available
tables = con.execute('SHOW TABLES').fetchdf()['name'].tolist()
print('Tables in DuckDB:', tables)
assert 'model_features' in tables, 'ERROR: model_features not found — run notebooks 01, 02, 03, 03b first'


# Step 4: Data Preprocessing — Feature Engineering, Imputation, Encoding, and Train/Test Split


In [ ]:
import pandas as pd
import numpy as np

df = con.execute("SELECT * FROM model_features").fetchdf()
print(f"Loaded model_features: {df.shape[0]} rows, {df.shape[1]} columns")

# ── Confirm medication features are present ───────────────────────────────────
MED_COLS = [
    'on_loop_diuretic', 'on_ace_arb', 'on_beta_blocker',
    'on_aldosterone_antagonist', 'on_digoxin', 'on_anticoagulant',
    'num_unique_drugs', 'furosemide_max_dose', 'gdmt_score'
]
found     = [c for c in MED_COLS if c in df.columns]
not_found = [c for c in MED_COLS if c not in df.columns]
print(f"\nMedication features found:     {len(found)} / {len(MED_COLS)}")
if not_found:
    print(f"WARNING — missing cols (run 03b first): {not_found}")

# ── 1. Add lab CHANGE features (last - first) ─────────────────────────────────
LABS = [
    'creatinine', 'urea_nitrogen', 'sodium', 'potassium', 'glucose',
    'hemoglobin', 'white_blood_cells', 'platelet_count', 'bicarbonate',
    'calcium_total', 'inrpt', 'ptt', 'troponin_t',
    'creatine_kinase_mb_isoenzyme'
]
for lab in LABS:
    df[f'{lab}_change'] = df[f'{lab}_last'] - df[f'{lab}_first']

print(f"\nAfter adding lab change features: {df.shape[0]} rows, {df.shape[1]} columns")

# ── 2. Missing value imputation ───────────────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for drop_col in ['readmitted_30d', 'hadm_id', 'subject_id']:
    if drop_col in num_cols:
        num_cols.remove(drop_col)

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

cat_cols = ['gender', 'admission_type', 'insurance', 'marital_status', 'race', 'discharge_location']
for col in cat_cols:
    df[col] = df[col].fillna('UNKNOWN')

print(f"Missing values after imputation: {df.isnull().sum().sum()}")

# ── 3. One-hot encode categoricals ───────────────────────────────────────────
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print(f"After encoding: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns")

# ── 4. Train/Test Split ───────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

drop_cols = ['hadm_id', 'subject_id', 'readmitted_30d']
X = df_encoded.drop(columns=drop_cols)
y = df_encoded['readmitted_30d']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n{'='*40}")
print(f"FINAL SPLIT SUMMARY")
print(f"{'='*40}")
print(f"Training set : {X_train.shape[0]} rows x {X_train.shape[1]} features")
print(f"Test set     : {X_test.shape[0]} rows x {X_test.shape[1]} features")
print(f"\nFeature count breakdown:")
print(f"  Original features (pre-meds) : 137")
print(f"  Medication features added    : {len(found)}")
print(f"  Total features now           : {X_train.shape[1]}")
print(f"\nTrain label distribution:")
print(y_train.value_counts().to_string())
print(f"\nTest label distribution:")
print(y_test.value_counts().to_string())

# ── 5. Save splits to DuckDB ──────────────────────────────────────────────────
con.execute("CREATE OR REPLACE TABLE X_train AS SELECT * FROM X_train")
con.execute("CREATE OR REPLACE TABLE X_test  AS SELECT * FROM X_test")
y_train_df = y_train.to_frame()
y_test_df  = y_test.to_frame()
con.execute("CREATE OR REPLACE TABLE y_train AS SELECT * FROM y_train_df")
con.execute("CREATE OR REPLACE TABLE y_test  AS SELECT * FROM y_test_df")
print("\nSaved X_train, X_test, y_train, y_test to DuckDB")
print("Ready for: 05_Model_Training.ipynb")


Prepares the dataset for modeling through four preprocessing steps:
1. **Lab change features** — computes `last - first` values for all 14 labs to capture patient trajectory during hospitalization.
2. **Missing value imputation** — fills numerical columns with median values and categorical columns with 'UNKNOWN'.
3. **One-hot encoding** — converts 6 categorical features (gender, admission type, insurance, marital status, race, discharge location) into binary columns using `pd.get_dummies`.
4. **Stratified train/test split** — 80/20 split preserving the class ratio (~21.5% readmitted in both sets).

Final dimensions: Training set (3,606 rows × 137 features), Test set (902 rows × 137 features).

In [3]:
con.close()